# LLM Playground on Isambard

Run **Nemotron 3 Super (120B-A12B)** across the four GH200s of one Isambard node.

Set up first (see `README.md`): `bash setup_environment.sh` on a login node,
`sbatch tunnel.sbatch`, then connect and pick the `.venv` kernel.

## 1. Environment check

In [ ]:
import os

# LD_PRELOAD pointing at the system NCCL breaks `import torch`. It cannot be
# fixed here -- the loader applies it at exec time -- so detect and stop.
# tunnel.sbatch unsets it for you. (README, trap 2.)
if os.environ.get("LD_PRELOAD"):
    raise SystemExit(
        f"LD_PRELOAD is set:\n    {os.environ['LD_PRELOAD']}\n\n"
        "Run `unset LD_PRELOAD` in the shell, then restart the kernel."
    )

# Where model weights are cached. Point PROJECT_STORAGE at your project's
# storage -- $HOME quotas are far too small for a 230 GiB model.
PROJECT_STORAGE = (
    os.environ.get("PROJECT_STORAGE")
    or os.environ.get("PROJECTDIR")
    or os.path.expanduser("~")
)
os.environ.setdefault("HF_HOME", os.path.join(PROJECT_STORAGE, "hf"))
print(f"HF_HOME = {os.environ['HF_HOME']}")

import torch, transformers

print(f"torch          {torch.__version__}")
print(f"transformers   {transformers.__version__}")
print(f"CUDA available {torch.cuda.is_available()}")
print(f"GPUs visible   {torch.cuda.device_count()}")

# On ARM, plain `pip install torch` gives a CPU-only or CUDA-13 wheel, both of
# which fail silently or late. pyproject.toml pins cu126. (README, trap 1.)
assert "+cu" in torch.__version__, f"{torch.__version__}: CPU-only wheel, see pyproject.toml"
assert torch.cuda.is_available(), "No GPU -- login node, or job lacks --gpus-per-node=4?"

total = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {p.name}  sm_{p.major}{p.minor}  {p.total_memory/2**30:.1f} GiB")
print(f"\naggregate GPU memory: {total/2**30:.1f} GiB")

## 2. Load the model

~225 GiB of weights, so it needs 4 GPUs (380 GiB aggregate). `device_map="auto"`
shards it: one process, layers spread across GPUs, activations copied between
them. No NCCL, no process group — which is why no NCCL module is loaded.

Takes ~110 s from Lustre. The "fast path is not available" warning is expected:
the Mamba2 CUDA kernels have no ARM wheels, so `transformers` uses a correct
pure-PyTorch fallback.

In [ ]:
import time
from transformers import pipeline

MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

t0 = time.time()
pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype=torch.bfloat16,   # note: `dtype`, not the older `torch_dtype`
    device_map="auto",      # shard across all visible GPUs
)
print(f"loaded in {time.time()-t0:.1f}s")

In [ ]:
from collections import Counter

# Which GPU did each layer land on?
spread = Counter(pipe.model.hf_device_map.values())
print("layers per GPU:", dict(sorted(spread.items())))

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB used / {total/2**30:.1f} GiB")

## 3. Generating text

This is a reasoning model and reasoning is **on by default** — a short prompt
will spend its whole budget inside a `<think>` block and never answer. Turn it
off with `enable_thinking=False`, which is a *chat template* argument, not a
`generate` argument: passing it to the pipeline raises `ValueError`. So render
the prompt first, then generate from the string.

Two harmless warnings appear on every call (a stale `max_length=20` in the
model's `generation_config`, and a `transformers` deprecation notice).

In [ ]:
def chat(user_message, max_new_tokens=128, think=False, **kw):
    """Render the chat template, then generate. Returns the assistant's reply."""
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=think,      # a CHAT TEMPLATE kwarg, not a generate kwarg
    )
    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,             # reply only, not the prompt back
        clean_up_tokenization_spaces=False, # destructive for BPE tokenizers
        **kw,
    )
    return out[0]["generated_text"].strip()

### Example 1 — a short factual explanation

In [ ]:
t0 = time.time()
reply = chat("Explain what a supercomputer interconnect does, in two sentences.")
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

### Example 2 — something creative

In [ ]:
t0 = time.time()
reply = chat("Write a haiku about GPUs waiting in a job queue.", max_new_tokens=64)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

### Example 3 — reasoning switched on

Same helper, `think=True`. Watch the model narrate its approach inside a
`<think>` block before answering.

In [ ]:
t0 = time.time()
reply = chat(
    "A job needs 12 GPUs. Each node has 4. How many nodes, and what if one node fails?",
    max_new_tokens=200,
    think=True,
)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

## 4. Freeing the GPUs

In [ ]:
import gc

# Dropping the reference is necessary but often not sufficient: Jupyter keeps
# its own references to cell outputs, so memory may not fall to zero here.
# Restarting the kernel is the reliable way to fully release the GPUs.
del pipe
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB still resident")

print("\nIf that is still large, restart the kernel to fully free the GPUs.")

---

**Next:** jobs are capped at 24 h — save to project storage, not the node.
Models too large for one node (the 550B Ultra, ~1.1 TB) need multi-node serving
with vLLM, where the Slingshot and NCCL setup skipped here starts to matter.